# DataStore Magic Commands — Feature Test

This notebook tests all `%%save` and `%%query` magic features using synthetic data.
No external database connections are required.

**Features tested:**
- `%%save` — SQL mode (DuckDB), `--file`, `--file --polars`, `--df`, `--remote` (mocked)
- `%%save` — glob patterns (`*`), `--mode` (error/replace/append), `--chunk-size`, `--compression`
- `%%query` — `--var`, `--polars`, `--dql`, `--create`, `--persist`
- `%%dql` — bare (magic-duckdb), `--var`, `--polars`, `--create`, `--persist`
- `%%parqstore` — `--save` and `--query` unified syntax
- `store.list()`, `store.drop()`, `store.clear_all()`

---
## 0. Setup — generate test CSV/Excel/Parquet files

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import os, shutil

# Working directories
TEST_DIR = "./test_magic_data"
STORE_DIR = "./test_magic_store"

# Clean up from previous runs
for d in [TEST_DIR, STORE_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

np.random.seed(42)
N = 50_000

# --- customers.csv ---
customers = pd.DataFrame({
    "cust_id": range(1, 1001),
    "name": [f"Customer_{i}" for i in range(1, 1001)],
    "region": np.random.choice(["APAC", "EMEA", "AMER"], 1000),
    "segment": np.random.choice(["Enterprise", "SMB", "Startup"], 1000),
})
customers.to_csv(f"{TEST_DIR}/customers.csv", index=False)

# --- transactions split into 3 CSV files (for glob test) ---
for i, month in enumerate(["01", "02", "03"]):
    chunk = pd.DataFrame({
        "txn_id": range(i * N, (i + 1) * N),
        "cust_id": np.random.randint(1, 1001, N),
        "product": np.random.choice(["Widget", "Gadget", "Doohickey", "Thingamajig"], N),
        "amount": np.round(np.random.exponential(500, N), 2),
        "txn_date": pd.date_range(f"2024-{month}-01", periods=N, freq="min"),
    })
    chunk.to_csv(f"{TEST_DIR}/transactions_{month}.csv", index=False)

# --- products.xlsx ---
products = pd.DataFrame({
    "product": ["Widget", "Gadget", "Doohickey", "Thingamajig"],
    "category": ["Electronics", "Electronics", "Hardware", "Hardware"],
    "unit_cost": [120.0, 85.5, 45.0, 200.0],
})
products.to_excel(f"{TEST_DIR}/products.xlsx", index=False, sheet_name="Products")

# --- events.parquet ---
events = pd.DataFrame({
    "event_id": range(10_000),
    "cust_id": np.random.randint(1, 1001, 10_000),
    "event_type": np.random.choice(["click", "view", "purchase", "refund"], 10_000),
    "ts": pd.date_range("2024-01-01", periods=10_000, freq="5min"),
})
events.to_parquet(f"{TEST_DIR}/events.parquet", index=False)

# --- pipe-delimited file with odd extension (for --file-type test) ---
customers.to_csv(f"{TEST_DIR}/customers.dat", index=False, sep="|")

print("Test files created:")
for f in sorted(os.listdir(TEST_DIR)):
    size = os.path.getsize(os.path.join(TEST_DIR, f))
    print(f"  {f:30s}  {size:>10,} bytes")

---
## 1. Initialize DataStore + enable magics

In [ ]:
from datasuite.parquet_store import DataStore

store = DataStore(STORE_DIR)
store.enable_magic()

---
## 2. `%%save --file` — single CSV file

In [ ]:
%%save customers --file
./test_magic_data/customers.csv

In [ ]:
store.list()

## 3. `%%save --file` — glob pattern (multiple CSV files)

In [ ]:
%%save transactions --file
./test_magic_data/transactions_*.csv

In [ ]:
store.list()

## 4. `%%save --file --polars` — Polars batched CSV reader

In [ ]:
%%save transactions_pl --file --polars
./test_magic_data/transactions_*.csv

In [ ]:
# Compare row counts — should be identical
print("pandas reader:", store.info("transactions")["rows"])
print("polars reader:", store.info("transactions_pl")["rows"])

## 5. `%%save --file` — Excel file

In [ ]:
%%save products --file --file-type xlsx
./test_magic_data/products.xlsx
sheet_name=Products

## 6. `%%save --file` — Parquet file

In [ ]:
%%save events --file
./test_magic_data/events.parquet

## 7. `%%save --file` — force file type with `--file-type`

In [ ]:
%%save customers_pipe --file --file-type csv
./test_magic_data/customers.dat
sep=|

## 8. `%%save --file` — custom chunk-size and compression

In [ ]:
%%save events_gzip --file --chunk-size 3000 --compression gzip
./test_magic_data/events.parquet

In [ ]:
# Should have more chunks due to smaller chunk-size
info = store.info("events_gzip")
print(f"chunks: {info['chunks']}, compression: {info['compression']}")

## 9. `%%save --df` — save a notebook DataFrame variable

In [ ]:
# Create a Polars LazyFrame in the notebook namespace
lf_regions = pl.LazyFrame({
    "region": ["APAC", "EMEA", "AMER"],
    "timezone": ["Asia/Bangkok", "Europe/London", "America/New_York"],
    "currency": ["THB", "EUR", "USD"],
})
lf_regions.collect()

In [ ]:
%save regions --df lf_regions

## 10. `%%save` — SQL mode (DuckDB, no `--conn`)

In [ ]:
%%save top_customers
SELECT c.cust_id, c.name, c.region, SUM(t.amount) AS total_spend
FROM transactions t
JOIN customers c USING (cust_id)
GROUP BY c.cust_id, c.name, c.region
ORDER BY total_spend DESC
LIMIT 100

## 11. `%%save --mode replace` — overwrite existing dataset

In [ ]:
%%save top_customers --mode replace
SELECT c.cust_id, c.name, c.region, c.segment, SUM(t.amount) AS total_spend
FROM transactions t
JOIN customers c USING (cust_id)
GROUP BY c.cust_id, c.name, c.region, c.segment
ORDER BY total_spend DESC
LIMIT 200

In [ ]:
store.info("top_customers")["rows"]  # should be 200

## 12. `%%save --mode append` — add rows to existing dataset

In [ ]:
%%save events_incremental --file
./test_magic_data/events.parquet

In [ ]:
%%save events_incremental --file --mode append
./test_magic_data/events.parquet

In [ ]:
# Should have 20,000 rows (10k + 10k)
print("rows:", store.info("events_incremental")["rows"])

## 13. `%%save --mode error` — should fail on existing dataset

In [ ]:
try:
    store.save_file("customers", "./test_magic_data/customers.csv", mode="error")
except FileExistsError as e:
    print(f"Expected error: {e}")

---
## 14. Verify all datasets loaded

In [ ]:
store.list()

---
## 15. `%%query` — basic query (display only)

In [ ]:
%%query
SELECT * FROM customers LIMIT 5

## 16. `%%query --var` — store result in variable

In [ ]:
%%query --var df_summary
SELECT
    c.region,
    c.segment,
    COUNT(*) AS txn_count,
    ROUND(SUM(t.amount), 2) AS total_amount,
    ROUND(AVG(t.amount), 2) AS avg_amount
FROM transactions t
JOIN customers c USING (cust_id)
GROUP BY c.region, c.segment
ORDER BY total_amount DESC

In [ ]:
# Variable should be accessible
print(type(df_summary))
df_summary.head()

## 17. `%%query --var --polars` — store as LazyFrame

In [ ]:
%%query --var lf_products --polars
SELECT
    p.product,
    p.category,
    p.unit_cost,
    COUNT(*) AS times_sold,
    ROUND(SUM(t.amount), 2) AS revenue
FROM transactions t
JOIN products p ON t.product = p.product
GROUP BY p.product, p.category, p.unit_cost

In [ ]:
# Should be a LazyFrame
print(type(lf_products))
lf_products.collect()

## 18. `%%query --dql` — render with magic-duckdb

In [ ]:
%%query --dql
SELECT region, COUNT(*) AS cnt
FROM customers
GROUP BY region
ORDER BY cnt DESC

## 19. `%%query --create` — create DuckDB in-memory table

In [ ]:
%%query --create high_value_txns --dql
SELECT t.*, c.name, c.region
FROM transactions t
JOIN customers c USING (cust_id)
WHERE t.amount > 2000

In [ ]:
# The created table is queryable in subsequent cells
%%query --dql
SELECT region, COUNT(*) AS cnt, ROUND(AVG(amount), 2) AS avg_amt
FROM high_value_txns
GROUP BY region

## 20. `%%query --persist` — save to chunked Parquet + register view

In [ ]:
%%query --persist monthly_summary --dql
SELECT
    DATE_TRUNC('month', t.txn_date) AS month,
    c.region,
    COUNT(*) AS txn_count,
    ROUND(SUM(t.amount), 2) AS total_amount
FROM transactions t
JOIN customers c USING (cust_id)
GROUP BY 1, 2
ORDER BY 1, 2

In [ ]:
# Should appear in store.list() (persisted to disk)
store.list()

## 21. `%%query --persist --mode replace` — overwrite persisted dataset

In [ ]:
%%query --persist monthly_summary --mode replace
SELECT
    DATE_TRUNC('month', t.txn_date) AS month,
    c.region,
    c.segment,
    COUNT(*) AS txn_count,
    ROUND(SUM(t.amount), 2) AS total_amount,
    ROUND(AVG(t.amount), 2) AS avg_amount
FROM transactions t
JOIN customers c USING (cust_id)
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

---
## 22. `%%dql` — bare (original magic-duckdb)

In [ ]:
%%dql
SELECT product, COUNT(*) AS cnt, ROUND(SUM(amount), 2) AS total
FROM transactions
GROUP BY product
ORDER BY total DESC

## 23. `%%dql --var` — capture result

In [ ]:
%%dql --var df_region_totals
SELECT region, ROUND(SUM(amount), 2) AS total
FROM transactions t
JOIN customers c USING (cust_id)
GROUP BY region

In [ ]:
print(type(df_region_totals))
df_region_totals

## 24. `%%dql --polars --var` — capture as LazyFrame

In [ ]:
%%dql --polars --var lf_events
SELECT event_type, COUNT(*) AS cnt
FROM events
GROUP BY event_type

In [ ]:
print(type(lf_events))
lf_events.collect()

## 25. `%%dql --create` — create DuckDB table

In [ ]:
%%dql --create purchase_events
SELECT * FROM events WHERE event_type = 'purchase'

In [ ]:
%%dql
SELECT COUNT(*) AS purchase_count FROM purchase_events

## 26. `%%dql --persist` — persist to Parquet

In [ ]:
%%dql --persist customer_360
SELECT
    c.cust_id,
    c.name,
    c.region,
    c.segment,
    r.timezone,
    r.currency,
    COUNT(DISTINCT t.txn_id) AS num_txns,
    ROUND(SUM(t.amount), 2) AS total_spend,
    COUNT(DISTINCT e.event_id) AS num_events
FROM customers c
LEFT JOIN regions r USING (region)
LEFT JOIN transactions t USING (cust_id)
LEFT JOIN events e USING (cust_id)
GROUP BY 1, 2, 3, 4, 5, 6

In [ ]:
store.info("customer_360")

---
## 27. `%%parqstore --save` — unified syntax

In [ ]:
%%parqstore --save product_revenue
SELECT
    p.product,
    p.category,
    ROUND(SUM(t.amount), 2) AS revenue
FROM transactions t
JOIN products p ON t.product = p.product
GROUP BY 1, 2

## 28. `%%parqstore --save --file`

In [ ]:
%%parqstore --save customers_v2 --file --mode replace
./test_magic_data/customers.csv

## 29. `%%parqstore --query`

In [ ]:
%%parqstore --query --var df_rev
SELECT * FROM product_revenue ORDER BY revenue DESC

In [ ]:
df_rev

## 30. `%%parqstore --query --persist --dql` — full combo

In [ ]:
%%parqstore --query --persist region_kpi --dql
SELECT
    c.region,
    r.currency,
    COUNT(DISTINCT c.cust_id) AS num_customers,
    COUNT(*) AS num_txns,
    ROUND(SUM(t.amount), 2) AS total_revenue,
    ROUND(AVG(t.amount), 2) AS avg_ticket
FROM transactions t
JOIN customers c USING (cust_id)
JOIN regions r USING (region)
GROUP BY 1, 2

---
## 31. Export to Excel

In [ ]:
from datasuite import export_excel

export_excel("./test_magic_report.xlsx", [
    {"sheetname": "Summary",  "dataframe": df_summary},
    {"sheetname": "Revenue",  "dataframe": df_rev},
    {"sheetname": "Regions",  "dataframe": df_region_totals},
])
print("Saved test_magic_report.xlsx")

---
## 32. Final catalog

In [ ]:
store.list()

## 33. `store.drop()` — remove a single dataset

In [ ]:
store.drop("transactions_pl", files=True)
store.drop("events_gzip", files=True)
store.list()

## 34. `store.clear_all()` — remove everything

In [ ]:
store.clear_all(files=True)
store.list()

---
## 35. Cleanup test files

In [ ]:
import shutil
shutil.rmtree("./test_magic_data", ignore_errors=True)
shutil.rmtree("./test_magic_store", ignore_errors=True)
if os.path.exists("./test_magic_report.xlsx"):
    os.remove("./test_magic_report.xlsx")
print("Cleanup complete.")